**Input Tables: driver_telemetry_data, policy, vehicle,claim, accident**

**Output Tables: gold_trip_features, gold_policy_period_features, gold_policy_period_loss, gold_expected_loss_scores, gold_policy_premium_recommendation, gold_premium_reason_codes**

In [1]:
# PARAMETERS
import os

AS_OF_DATE = "2025-12-31"          # scoring cut-off
TARGET_LOSS_RATIO = 0.65
EXPENSE_LOAD = 0.15
PROFIT_LOAD = 0.05
MAX_CHANGE_PCT = 0.15              # +/- 15% cap
SMOOTHING_ALPHA = 0.3              # 30% new, 70% old (or invert as you prefer)
FEATURE_VERSION = "v1"

# Azure ML Configuration - Update these with your endpoint details
AZURE_ML_ENDPOINT_URL = os.environ.get("AZURE_ML_ENDPOINT_URL", "https://ubi-risk-endpoint.eastus2.inference.ml.azure.com/score")  # From Step 10 of azureml_train_deploy_model.ipynb
AZURE_ML_ENDPOINT_KEY = os.environ.get("AZURE_ML_ENDPOINT_KEY", "C73Ffb4NEWGBhfzJW2zywiqABonA0ceO2rlZYrR9ZnfRO7v50V4GJQQJ99CCAAAAAAAAAAAAINFRAZML3kWc")  # Primary key from Step 10
MODEL_NAME = os.environ.get("AZURE_ML_MODEL_NAME", "azure_ml_ubi_model")
MODEL_VERSION = os.environ.get("AZURE_ML_MODEL_VERSION", "1")

# Validate required Azure ML configurations
if not AZURE_ML_ENDPOINT_URL or not AZURE_ML_ENDPOINT_KEY:
    print("⚠️  Warning: Azure ML endpoint configuration not set!")
    print("   Required: AZURE_ML_ENDPOINT_URL and AZURE_ML_ENDPOINT_KEY")
    print("   Get these from Step 10 output of azureml_train_deploy_model.ipynb")
else:
    print(f"✓ Azure ML Configuration loaded:")
    print(f"  Model: {MODEL_NAME} v{MODEL_VERSION}")
    print(f"  Endpoint URL: {AZURE_ML_ENDPOINT_URL[:50]}...")
    print(f"  API Key: {'*' * 8}{AZURE_ML_ENDPOINT_KEY[-4:]}")

StatementMeta(, 311abca1-4509-4b60-8e98-df1f0920d3ba, 3, Finished, Available, Finished, False)

✓ Azure ML Configuration loaded:
  Model: azure_ml_ubi_model v1
  Endpoint URL: https://ubi-risk-endpoint.eastus2.inference.ml.azu...
  API Key: ********3kWc


In [2]:
#Get all tables data
trips = spark.table("driver_telemetry_data")
policy = spark.table("policy")
vehicle = spark.table("vehicle")
claims = spark.table("claim")
acc = spark.table("accident")

StatementMeta(, 311abca1-4509-4b60-8e98-df1f0920d3ba, 4, Finished, Available, Finished, False)

**Build gold_trip_features**
- Goal: normalize and compute per-trip derived rates.

In [3]:
from pyspark.sql import functions as F

trip_features = (trips
  .withColumn("trip_date", F.to_date("start_time"))
  .withColumn("is_night", F.col("time_of_day_category") == F.lit("Night"))
  .withColumn("harsh_events_total", F.col("sudden_braking_count") + F.col("rapid_acceleration_count") + F.col("harsh_cornering_count"))
  .withColumn("harsh_events_per_100_miles",
              F.when(F.col("distance_miles") > 0, 100.0*F.col("harsh_events_total")/F.col("distance_miles")).otherwise(None))
  .withColumn("speeding_per_100_miles",
              F.when(F.col("distance_miles") > 0, 100.0*F.col("speeding_incidents")/F.col("distance_miles")).otherwise(None))
  .withColumn("feature_version", F.lit(FEATURE_VERSION))
  .withColumn("ingest_ts", F.current_timestamp())
)

trip_features.write.mode("overwrite").format("delta").saveAsTable("gold_trip_features")

StatementMeta(, 311abca1-4509-4b60-8e98-df1f0920d3ba, 5, Finished, Available, Finished, False)

df = spark.sql("SELECT * FROM lh_AutoClaims.dbo.gold_trip_features LIMIT 1000")
display(df)

**Create Trip → Policy mapping (policy-period alignment)**
- Critical UBI step: map each trip to the active policy term for that driver+vehicle.

In [4]:
# join on policyholder_id + vin and trip within policy dates
trip_policy = (trip_features.alias("t")
  .join(policy.alias("p"),
        (F.col("t.policyholder_id") == F.col("p.policyholder_id")) &
        (F.col("t.vin") == F.col("p.vehicle_vin")) &
        (F.col("t.trip_date") >= F.col("p.start_date")) &
        (F.col("t.trip_date") <  F.col("p.end_date")),
        "inner")
  .select(
      F.col("p.policy_number"),
      F.col("p.coverage_type"),
      F.col("p.start_date").alias("policy_start_date"),
      F.col("p.end_date").alias("policy_end_date"),
      F.col("p.premium").alias("current_premium"),
      F.col("t.*")
  )
)
trip_policy.createOrReplaceTempView("trip_policy")

StatementMeta(, 311abca1-4509-4b60-8e98-df1f0920d3ba, 6, Finished, Available, Finished, False)

%%sql
select * from trip_policy order by policy_number

**Build gold_policy_period_features (aggregate per policy term)**

In [5]:
%%sql
CREATE OR REPLACE TEMP VIEW policy_period_features AS
SELECT
  policy_number,
  policyholder_id,
  vin AS vehicle_vin,
  coverage_type,
  policy_start_date,
  policy_end_date,
  FIRST(current_premium) AS current_premium,

  COUNT(*) AS total_trips,
  SUM(distance_miles) AS total_miles,
  AVG(safety_score) AS avg_safety_score,

  SUM(CASE WHEN is_night THEN distance_miles ELSE 0 END) / NULLIF(SUM(distance_miles),0) AS night_miles_share,

  (100.0 * SUM(speeding_incidents) / NULLIF(SUM(distance_miles),0)) AS speeding_per_100_miles,
  (100.0 * SUM(harsh_events_total) / NULLIF(SUM(distance_miles),0)) AS harsh_events_per_100_miles,

  AVG(CASE WHEN trip_risk_level='High' THEN 1 ELSE 0 END) AS high_risk_trip_share,

  'v1' AS feature_version,
  current_timestamp() AS computed_ts
FROM trip_policy
GROUP BY
  policy_number, policyholder_id, vin, coverage_type, policy_start_date, policy_end_date;

StatementMeta(, 311abca1-4509-4b60-8e98-df1f0920d3ba, 7, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

%%sql
select * from policy_period_features order by policy_number

In [6]:
ppf = spark.sql("SELECT * FROM policy_period_features")
ppf = (ppf.join(vehicle, ppf.vehicle_vin == vehicle.vin, "left")
          .drop(vehicle.vin)
          .withColumnRenamed("year","vehicle_year")
          .withColumnRenamed("make","vehicle_make")
          .withColumnRenamed("model","vehicle_model")
)
ppf.write.mode("overwrite").format("delta").saveAsTable("gold_policy_period_features")

StatementMeta(, 311abca1-4509-4b60-8e98-df1f0920d3ba, 8, Finished, Available, Finished, False)

**Compute outcomes (optional but recommended for calibration): gold_policy_period_loss**
- If you want to validate whether your premiums are adequate, compute loss by policy period.

In [7]:
# Claims joined by policy_number and date_filed in period
claims_in_period = (claims.alias("c")
  .join(policy.alias("p"), F.col("c.policy_number") == F.col("p.policy_number"), "inner")
  .where((F.col("c.date_filed") >= F.col("p.start_date")) & (F.col("c.date_filed") < F.col("p.end_date")))
  .groupBy("c.policy_number", F.col("p.start_date").alias("policy_start_date"), F.col("p.end_date").alias("policy_end_date"))
  .agg(
      F.count("*").alias("claims_count"),
      F.sum("claim_amount").alias("total_claim_amount"),
      F.sum("payout_amount").alias("total_payout_amount")
  )
)

# Accident severity counts in period (optional)
acc_in_period = (acc.alias("a")
  .join(policy.alias("p"),
        (F.col("a.policyholder_id")==F.col("p.policyholder_id")) &
        (F.col("a.vehicle_vin")==F.col("p.vehicle_vin")) &
        (F.col("a.accident_date") >= F.col("p.start_date")) & (F.col("a.accident_date") < F.col("p.end_date")),
        "inner")
  .groupBy("p.policy_number", F.col("p.start_date").alias("policy_start_date"), F.col("p.end_date").alias("policy_end_date"))
  .agg(
      F.count("*").alias("accidents_count"),
      F.sum(F.when(F.col("severity")==F.lit("High"),1).otherwise(0)).alias("high_severity_accidents")
  )
)

loss = (claims_in_period.join(acc_in_period, ["policy_number","policy_start_date","policy_end_date"], "left")
        .fillna(0, ["accidents_count","high_severity_accidents"])
        .withColumn("label_version", F.lit("v1"))
        .withColumn("computed_ts", F.current_timestamp())
)

loss.write.mode("overwrite").format("delta").saveAsTable("gold_policy_period_loss")

StatementMeta(, 311abca1-4509-4b60-8e98-df1f0920d3ba, 9, Finished, Available, Finished, False)

%%sql
select * from gold_policy_period_loss

**Compute Expected Loss Cost (ELC) — rules-based MVP (fast + explainable)**
- Here’s a strong MVP approach: calibrate a baseline expected loss by coverage_type, then apply a telematics risk multiplier.


**Baseline loss cost by coverage (from historical payouts)**

In [8]:
loss = spark.table("gold_policy_period_loss")
ppf  = spark.table("gold_policy_period_features")

baseline = (ppf.join(loss, ["policy_number","policy_start_date","policy_end_date"], "left")
  .groupBy("coverage_type")
  .agg(F.avg("total_payout_amount").alias("baseline_elc"))
  .fillna(0, ["baseline_elc"])
)
baseline.createOrReplaceTempView("baseline_elc")

StatementMeta(, 311abca1-4509-4b60-8e98-df1f0920d3ba, 10, Finished, Available, Finished, False)

**Azure ML Model Integration - Risk Scoring**

This section replaces the previous rules-based scoring with predictions from an Azure ML online endpoint.

**Model Input Features:**
- Policy period aggregated features (speeding, harsh events, night miles, safety score)
- Baseline expected loss cost by coverage type
- Trip statistics (total trips, miles, high-risk trip share)

**Model Output (Expected):**
- `risk_factor`: Multiplier for baseline loss cost (e.g., 1.0 = average, 1.5 = 50% higher risk)
- `expected_loss_cost`: Predicted cost of claims for this policy
- `risk_score`: Normalized risk score (0-100 scale)

**Requirements:**
1. Azure ML endpoint must be deployed and accessible
2. Update parameters in first cell with your Azure ML details
3. Ensure model input schema matches the features sent
4. Run `az login` before executing if using DefaultAzureCredential

In [9]:
# Import required libraries for HTTP requests
import requests
import json
import pandas as pd

# Prepare features from gold_policy_period_features for Azure ML model
# The model expects these 8 features in this exact order
feature_columns = [
    'speeding_per_100_miles',
    'harsh_events_per_100_miles',
    'night_miles_share',
    'avg_safety_score',
    'total_trips',
    'total_miles',
    'high_risk_trip_share',
    'baseline_elc'
]

# Load features from gold table
ppf = spark.table("gold_policy_period_features")
baseline_tbl = spark.sql("SELECT * FROM baseline_elc")

# Join with baseline to get coverage-specific baseline loss cost
features_with_baseline = ppf.join(baseline_tbl, "coverage_type", "left")

# Select required features
features_df = features_with_baseline.select(
    "policy_number",
    "policy_start_date",
    "policy_end_date",
    *feature_columns
)

# Convert to Pandas for Azure ML endpoint invocation
features_pdf = features_df.toPandas()

print(f"✓ Loaded {len(features_pdf)} policy periods for scoring")
print(f"  Features: {', '.join(feature_columns)}")

# Create input payload for Azure ML endpoint
input_data = features_pdf[feature_columns].fillna(0).to_dict(orient='records')
payload = {"input_data": input_data}

# Call Azure ML endpoint for predictions via HTTP
print(f"\nCalling Azure ML endpoint with {len(input_data)} records...")

try:
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {AZURE_ML_ENDPOINT_KEY}"
    }
    
    response = requests.post(
        AZURE_ML_ENDPOINT_URL,
        json=payload,
        headers=headers,
        timeout=300  # 5 minute timeout for large batches
    )
    
    response.raise_for_status()  # Raise error for bad status codes
    predictions = response.json()
    
    print(f"✓ Received predictions from Azure ML")
    
    # Convert predictions to DataFrame
    predictions_df = pd.DataFrame(predictions)
    
    # Expected model output columns: risk_factor, expected_loss_cost, risk_score
    # Round to 2 decimal places
    predictions_df['risk_factor'] = predictions_df['risk_factor'].round(2)
    predictions_df['expected_loss_cost'] = predictions_df['expected_loss_cost'].round(2)
    predictions_df['risk_score'] = predictions_df['risk_score'].round(2)
    
    # Add predictions to features DataFrame
    features_pdf['risk_factor'] = predictions_df['risk_factor'].values
    features_pdf['expected_loss_cost'] = predictions_df['expected_loss_cost'].values
    features_pdf['risk_score'] = predictions_df['risk_score'].values
    
    print(f"✓ Added predictions to {len(features_pdf)} records")
    print(f"\nSample predictions:")
    print(features_pdf[['policy_number', 'risk_factor', 'expected_loss_cost', 'risk_score']].head())
    
except requests.exceptions.RequestException as e:
    print(f"❌ Error calling Azure ML endpoint: {e}")
    if hasattr(e, 'response') and e.response is not None:
        print(f"   Status code: {e.response.status_code}")
        print(f"   Response: {e.response.text[:500]}")
    raise
except Exception as e:
    print(f"❌ Error processing predictions: {e}")
    raise

# Convert back to Spark DataFrame
scores = spark.createDataFrame(features_pdf)

# Create scores temp view for next cell
scores.createOrReplaceTempView("expected_loss_scores")

print(f"\n✓ Successfully scored {scores.count()} policies using Azure ML endpoint")

StatementMeta(, 311abca1-4509-4b60-8e98-df1f0920d3ba, 11, Finished, Available, Finished, False)

✓ Loaded 82 policy periods for scoring
  Features: speeding_per_100_miles, harsh_events_per_100_miles, night_miles_share, avg_safety_score, total_trips, total_miles, high_risk_trip_share, baseline_elc

Calling Azure ML endpoint with 82 records...
✓ Received predictions from Azure ML
✓ Added predictions to 82 records

Sample predictions:
  policy_number  risk_factor  expected_loss_cost  risk_score
0       POL2011         0.52              611.38       100.0
1       POL2020         0.93             1827.02       100.0
2       POL4020         0.70             1230.77       100.0
3       POL4039         0.78             1189.01       100.0
4       POL4001         0.77             1996.42       100.0

✓ Successfully scored 82 policies using Azure ML endpoint


In [10]:
# Add model metadata and save to gold table
scores_final = scores \
  .withColumn("model_name", F.lit(MODEL_NAME)) \
  .withColumn("model_version", F.lit(MODEL_VERSION)) \
  .withColumn("feature_version", F.lit(FEATURE_VERSION)) \
  .withColumn("scoring_ts", F.current_timestamp())

scores_final.write.mode("overwrite").format("delta").saveAsTable("gold_expected_loss_scores")

print(f"✓ Saved {scores_final.count()} scored policies to gold_expected_loss_scores table")
display(scores_final.limit(10))

StatementMeta(, 311abca1-4509-4b60-8e98-df1f0920d3ba, 12, Finished, Available, Finished, False)

✓ Saved 82 scored policies to gold_expected_loss_scores table


SynapseWidget(Synapse.DataFrame, 4da13dab-9eda-4e5a-b9d7-fccb07c529ed)

**Premium calculation (technical premium → guardrails → final)**

In [ ]:
# ATTENTION: AI-generated code can include errors or operations you didn't intend. Review the code in this cell carefully before running it.

# Import needed for disambiguation
from pyspark.sql import functions as F

scores = spark.table("gold_expected_loss_scores")
ppf = spark.table("gold_policy_period_features")
loss = spark.table("gold_policy_period_loss")

# To avoid column ambiguity after join (columns with the same name), we will rename overlapping columns in `scores` before join.
rename_cols = ["speeding_per_100_miles", "harsh_events_per_100_miles", "night_miles_share", "avg_safety_score", "feature_version"]
scores_renamed = scores
for col in rename_cols:
    scores_renamed = scores_renamed.withColumnRenamed(col, f"scores_{col}")

# Now join on keys. This avoids ambiguity without suffixes= param.
joined = (
    ppf.join(
        scores_renamed,
        on=["policy_number", "policy_start_date", "policy_end_date"],
        how="inner"
    )
    .withColumn("target_loss_ratio", F.lit(TARGET_LOSS_RATIO))
    .withColumn("expense_load", F.lit(EXPENSE_LOAD))
    .withColumn("profit_load", F.lit(PROFIT_LOAD))
    .drop("scores_feature_version")  # Remove the renamed 'feature_version' from scores
)

# Technical premium
joined = joined.withColumn(
    "recommended_technical_premium",
    (F.col("expected_loss_cost") / F.col("target_loss_ratio")) * (1.0 + F.col("expense_load") + F.col("profit_load"))
)

# Cap changes
joined = joined.withColumn("min_allowed", F.col("current_premium") * (1.0 - F.lit(MAX_CHANGE_PCT))) \
               .withColumn("max_allowed", F.col("current_premium") * (1.0 + F.lit(MAX_CHANGE_PCT))) \
               .withColumn("capped_premium",
                          F.when(F.col("recommended_technical_premium") < F.col("min_allowed"), F.col("min_allowed"))
                           .when(F.col("recommended_technical_premium") > F.col("max_allowed"), F.col("max_allowed"))
                           .otherwise(F.col("recommended_technical_premium"))) \
               .withColumn("cap_applied_flag",
                          (F.col("capped_premium") != F.col("recommended_technical_premium")))

# Smoothing
joined = joined.withColumn(
    "recommended_premium",
    (F.lit(SMOOTHING_ALPHA) * F.col("capped_premium")) + ((1.0 - F.lit(SMOOTHING_ALPHA)) * F.col("current_premium"))
).withColumn("smoothing_applied_flag", F.lit(True))

joined = joined.withColumn(
    "premium_change_pct",
    (F.col("recommended_premium") - F.col("current_premium")) / F.col("current_premium")
)

# Risk band derived from model risk_score (0-100). Bins are inclusive on the lower bound:
# 0-20 Very Low, 20-40 Low, 40-60 Medium, 60-80 High, 80-100 Very High.
joined = joined.withColumn(
    "risk_band",
    F.when(F.col("risk_score").isNull(), F.lit(None).cast("string"))
     .when(F.col("risk_score") < 20, F.lit("Very Low Risk"))
     .when(F.col("risk_score") < 40, F.lit("Low Risk"))
     .when(F.col("risk_score") < 60, F.lit("Medium Risk"))
     .when(F.col("risk_score") < 80, F.lit("High Risk"))
     .otherwise(F.lit("Very High Risk"))
)

# Reason codes (simple JSON string MVP) - use disambiguated scores columns
joined = joined.withColumn(
    "reason_codes",
    F.to_json(F.struct(
        F.col("scores_speeding_per_100_miles").alias("speeding_per_100_miles"),
        F.col("scores_harsh_events_per_100_miles").alias("harsh_events_per_100_miles"),
        F.col("scores_night_miles_share").alias("night_miles_share"),
        F.col("scores_avg_safety_score").alias("avg_safety_score")
    ))
).withColumn("scoring_ts", F.current_timestamp())

# -- Join with loss data to compute actual_loss_ratio & underpriced_flag --
joined = (
    joined
    .join(
        loss.select("policy_number", "policy_start_date", "policy_end_date", "total_payout_amount"),
        ["policy_number", "policy_start_date", "policy_end_date"],
        "left"
    )
    .withColumn(
        "actual_loss_ratio",
        F.when(F.col("recommended_premium") > 0,
               F.col("total_payout_amount") / F.col("recommended_premium"))
         .otherwise(F.lit(None))
    )
    .withColumn(
        "indicated_premium",
        F.when(F.col("target_loss_ratio") > 0,
               F.col("total_payout_amount") / F.col("target_loss_ratio"))
         .otherwise(F.lit(None))
    )
    .withColumn(
        "underpriced_flag",
        F.when(
            (F.col("indicated_premium").isNotNull()) &
            (F.col("recommended_premium") > 0) &
            ((F.col("indicated_premium") / F.col("recommended_premium")) > 1),
            F.lit("Y")
        ).otherwise(F.lit("N"))
    )
)

# Select output columns
premium_out = joined.select(
    "policy_number", "policy_start_date", "policy_end_date", "coverage_type",
    "current_premium", "expected_loss_cost",
    "risk_score", "risk_band",
    "target_loss_ratio", "expense_load", "profit_load",
    "recommended_technical_premium", "recommended_premium", "premium_change_pct",
    "cap_applied_flag", "smoothing_applied_flag",
    "actual_loss_ratio", "underpriced_flag",
    "reason_codes", "scoring_ts", "model_version", "feature_version"
)

premium_out.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("gold_policy_premium_recommendation")

StatementMeta(, 311abca1-4509-4b60-8e98-df1f0920d3ba, 13, Finished, Available, Finished, False)

**Add actual_loss_ratio and underpriced_flag columns to gold_policy_premium_recommendation table**

display(premium_out)

**(Optional) explode reason codes into gold_premium_reason_codes**

In [12]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, DoubleType

# If you stored reason_codes as JSON created from a struct in step 7:
schema = StructType([
    StructField("speeding_per_100_miles", DoubleType(), True),
    StructField("harsh_events_per_100_miles", DoubleType(), True),
    StructField("night_miles_share", DoubleType(), True),
    StructField("avg_safety_score", DoubleType(), True)
])

# premium_out must include reason_codes (your error message shows it does)
premium_out = spark.table("gold_policy_premium_recommendation")

rc_parsed = (premium_out
    .withColumn("rc", F.from_json(F.col("reason_codes"), schema))
    .select(
        "policy_number", "policy_start_date", "policy_end_date",
        F.col("rc.speeding_per_100_miles").alias("speeding_per_100_miles"),
        F.col("rc.harsh_events_per_100_miles").alias("harsh_events_per_100_miles"),
        F.col("rc.night_miles_share").alias("night_miles_share"),
        F.col("rc.avg_safety_score").alias("avg_safety_score"),
        "scoring_ts"
    )
)

# Convert wide -> long (factor rows) without needing explode(map)
reason_rows = (
    rc_parsed.selectExpr(
        "policy_number", "policy_start_date", "policy_end_date", "scoring_ts",
        "stack(4, "
        "'speeding_per_100_miles', speeding_per_100_miles, "
        "'harsh_events_per_100_miles', harsh_events_per_100_miles, "
        "'night_miles_share', night_miles_share, "
        "'avg_safety_score', avg_safety_score"
        ") as (factor_name, factor_value)"
    )
    .withColumn(
        "direction",
        F.when(F.col("factor_name") == F.lit("avg_safety_score"), F.lit("decreased"))
         .otherwise(F.lit("increased"))
    )
    .withColumn("contribution_score", F.lit(None).cast("double"))
    .withColumn("computed_ts", F.current_timestamp())
    .withColumn("reason_code_key",
        F.sha2(F.concat_ws("|",
            F.col("policy_number"),
            F.col("policy_start_date").cast("string"),
            F.col("policy_end_date").cast("string"),
            F.col("factor_name")), 256))
)

reason_rows.write \
    .mode("overwrite") \
    .format("delta") \
    .option("mergeSchema", "true") \
    .saveAsTable("gold_premium_reason_codes")

StatementMeta(, 311abca1-4509-4b60-8e98-df1f0920d3ba, 14, Finished, Available, Finished, False)

display(reason_rows)